# RL-Brand-Bound CO — End-to-End Demo

This notebook runs the four project stages in dummy mode so you can see what the system does end to end. Total runtime on CPU: roughly 3–5 minutes for the combinatorial auction problem.

Make sure you have already activated the `rl_bb` conda env and run `pip install -e .` from the repo root.

**Pipeline**
1. Stage 1 — Instance generation
2. Stage 2 — GCNN + SL (behavioral cloning on RB demonstrations)
3. Stage 3 — PPO+GAE training
4. Stage 4 — Evaluation: Random vs FSB vs PPO

## Setup

Load both YAML configs (base + dummy), seed everything, pick a device.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

from rl_bb.utils import configure_logging, load_config, resolve_device, resolve_path, set_seed

PROBLEM = "combinatorial_auction"
REPO = Path('..').resolve()

cfg = load_config(REPO / "config" / "base.yaml", REPO / "config" / "dummy.yaml")
set_seed(int(cfg["experiment"]["seed"]))
device = resolve_device(cfg["experiment"].get("device", "auto"))
log_dir = resolve_path(cfg["paths"]["log_dir"]) / cfg["experiment"]["name"]
configure_logging(log_dir=log_dir, filename="notebook.log")

print(f"experiment: {cfg['experiment']['name']}")
print(f"device:     {device}")
print(f"problem:    {PROBLEM}")

## Stage 1 — Instance Generation

Generates `.mps` files for every (regime, split) bucket. With dummy.yaml that's 20 train + 10 val + 10 test instances for the `train_size` regime, plus 10 + 10 for each transfer regime.

In [ ]:
from rl_bb.stage_1_instances import run_stage_1

summary = run_stage_1(cfg, problem=PROBLEM, smoke=False)
summary

In [ ]:
# How many .mps files landed on disk?
data_root = resolve_path(cfg["paths"]["data_dir"]) / cfg["experiment"]["name"] / PROBLEM
counts = {p.relative_to(data_root).parent.as_posix(): len(list(p.parent.glob('instance_*.mps')))
          for p in data_root.glob('*/*/instance_0000.mps')}
pd.Series(counts).rename("instances").to_frame()

## Stage 2 — GCNN + SL (Behavioral Cloning)

`run_stage_2` honors `config.pretrain.mode`:

- `auto` (default) — loads the cached checkpoint if it exists, otherwise collects RB demos and trains.
- `force_retrain` — always retrain and overwrite.
- `load_only` — require an existing checkpoint.

The first run on a fresh repo will train. Re-running this cell after success should print "cache hit" and skip training.

In [ ]:
from rl_bb.stage_2_pretrain import run_stage_2

out_s2 = run_stage_2(cfg, problem=PROBLEM)
print("mode:", out_s2["mode"])
print("ckpt:", out_s2["ckpt_path"])
if "best_val_loss" in out_s2:
    print(f"best val loss: {out_s2['best_val_loss']:.4f}")

In [ ]:
# Plot the training history if we just trained, or load it from disk if we hit cache.
ckpt_dir = resolve_path(cfg["paths"]["ckpt_dir"]) / cfg["experiment"]["name"]
history_path = ckpt_dir / "pretrain_history.json"
if history_path.exists():
    hist = pd.DataFrame(json.loads(history_path.read_text()))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(hist["epoch"], hist["train_loss"], label="train")
    if hist["val_loss"].notna().any():
        ax[0].plot(hist["epoch"], hist["val_loss"], label="val")
    ax[0].set(xlabel="epoch", ylabel="loss", title="BC cross-entropy")
    ax[0].legend()
    ax[1].plot(hist["epoch"], hist["train_top1"], label="train top-1")
    ax[1].plot(hist["epoch"], hist["train_top5"], label="train top-5")
    if hist["val_top1"].notna().any():
        ax[1].plot(hist["epoch"], hist["val_top1"], '--', label="val top-1")
        ax[1].plot(hist["epoch"], hist["val_top5"], '--', label="val top-5")
    ax[1].set(xlabel="epoch", ylabel="accuracy", title="Top-k action accuracy")
    ax[1].legend(); plt.tight_layout(); plt.show()
else:
    print('No history file (likely a cache hit on a freshly-cloned repo).')

## Stage 3 — PPO+GAE Training

Warm-starts from `pretrain_best.pt` and refines the policy with PPO. Dummy mode runs 3 iterations × 4 rollouts × 2 update epochs.

In [ ]:
from rl_bb.stage_3_ppo import run_stage_3

out_s3 = run_stage_3(cfg, problem=PROBLEM)
print(f"best mean reward: {out_s3['best_mean_reward']:.4f}")

In [ ]:
ppo_history = pd.DataFrame(out_s3['history'])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot(ppo_history["iter"], ppo_history["mean_reward"], marker='o')
ax[0].set(xlabel="iter", ylabel="mean reward", title="PPO reward")
if "policy_loss" in ppo_history.columns and ppo_history["policy_loss"].notna().any():
    ax[1].plot(ppo_history["iter"], ppo_history["policy_loss"], marker='o', label="policy")
    ax[1].plot(ppo_history["iter"], ppo_history["value_loss"], marker='o', label="value")
    ax[1].set(xlabel="iter", ylabel="loss", title="PPO losses"); ax[1].legend()
if "approx_kl" in ppo_history.columns and ppo_history["approx_kl"].notna().any():
    ax[2].plot(ppo_history["iter"], ppo_history["approx_kl"], marker='o', label="approx KL")
    ax[2].plot(ppo_history["iter"], ppo_history["clip_frac"], marker='o', label="clip frac")
    ax[2].set(xlabel="iter", title="PPO diagnostics"); ax[2].legend()
plt.tight_layout(); plt.show()

## Stage 4 — Evaluation

Random vs FSB vs PPO on the **test** split of all three regimes. With dummy.yaml we use 5 seeds × 10 test instances per regime. Restrict with `max_instances=3` to keep the notebook quick.

In [ ]:
from rl_bb.stage_4_eval import run_stage_4, write_results

per_instance, summary_rows = run_stage_4(
    cfg,
    problem=PROBLEM,
    split="test",
    policies=("random", "fsb", "ppo"),
    max_instances=3,
)
_ = write_results(per_instance, summary_rows, log_dir)
df = pd.DataFrame(summary_rows)
df[["policy", "regime", "split", "n", "n_nodes_mean", "n_nodes_std", "wall_time_s_mean"]]

In [ ]:
# Side-by-side bar chart: nodes per regime per policy.
pivot = df.pivot(index="regime", columns="policy", values="n_nodes_mean")
pivot = pivot.reindex(["train_size", "transfer_medium", "transfer_large"])
ax = pivot.plot(kind="bar", figsize=(8, 4))
ax.set(ylabel="mean B&B nodes", title=f"{PROBLEM}: nodes per regime per policy")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## What you should see

- **Stage 1:** ~60 `.mps` files per problem under `data/rl_bb_dummy/<problem>/`.
- **Stage 2:** Training loss decreasing, top-1 / top-5 climbing; checkpoint saved at `checkpoints/rl_bb_dummy/pretrain_best.pt`. Re-running uses the cache.
- **Stage 3:** Per-iteration log with reward, PPO losses, KL, clip fraction. With only a handful of iterations metrics are noisy; full mode (50 iters) stabilizes them.
- **Stage 4:** Random vs FSB vs PPO node counts per regime. On the dummy auction PPO typically matches FSB on transfer-size instances; Random uses many more nodes.